# DQN Pong group notebook

Alliane: environment and gameplay. Anthony: learning algorithm. HonourGod: repo, results and presentation.

We write and test our parts here first, then move them into `train.py` at the day-4 integration session.

How to use it:
1. Only edit your own section.
2. Keep the function names and arguments as they are. We agreed them on day 1 and the other sections depend on them.
3. Replace the `TODO` lines with your code and run the test cell below. It prints `PASS` when your part works, or `NOT WRITTEN YET` if the function is still empty.
4. When all the tests pass, run the smoke test in section 6. Nobody starts their 10 experiments before that.

## 1. Setup (everyone)
Runs locally with the project's `.venv` kernel, or on Google Colab. On Colab it clones the repo and installs the Atari packages with uv. Change `BRANCH` to your own branch.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/H-levison/dqn-atari.git"
    BRANCH = "alliane"  # change to your own branch
    if not os.path.exists("repo"):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "repo"], check=True)
    os.chdir("repo")
    # Colab already has torch, numpy, pandas and matplotlib
    subprocess.run(["uv", "pip", "install", "--system", "gymnasium", "ale-py", "opencv-python"], check=True)

print("Working directory:", os.getcwd())

In [ ]:
import copy
import csv
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

N_ACTIONS = 6  # Pong has 6 actions


def run_test(test_function):
    # Runs a test and prints the result. An empty function raises
    # NotImplementedError, which we report instead of crashing.
    try:
        test_function()
        print("PASS:", test_function.__name__)
    except NotImplementedError:
        print("NOT WRITTEN YET:", test_function.__name__)

## 2. Shared settings (everyone)
These are the values we agreed on day 1, so only change them if the whole group agrees. For your experiments, change hyperparameters with the `train.py` command-line flags instead.

In [ ]:
STEP_BUDGET = 400_000  # agent steps per experiment (400k steps = 1.6M frames with frame-skip 4)
CSV_COLUMNS = ["episode", "step", "reward", "length", "epsilon", "loss"]

default_config = {
    "lr": 1e-4,
    "gamma": 0.99,
    "batch_size": 32,
    "eps_start": 1.0,
    "eps_end": 0.02,
    "eps_decay": 100_000,  # number of steps for epsilon to go from eps_start down to eps_end
    "steps": STEP_BUDGET,
    "buffer_size": 100_000,  # uses about 5.6 GB of RAM
    "learning_starts": 10_000,  # random steps to fill the buffer before learning starts
    "train_every": 4,  # one learning update every 4 steps
    "target_update_every": 1_000,  # copy the online weights to the target network every 1000 steps
    "seed": 0,
    "out": "smoke_test/model.pth",  # model path; the CSV log is saved next to it
}

## 3. Environment, network and replay buffer (Alliane, done)
The code is in `environment.py`, `network.py` and `replay_buffer.py`, so `train.py` and `play.py` use the same version.

- `make_env(seed)` gives observations of shape `(4, 84, 84)` with dtype uint8.
- `DQN(n_actions)` takes a batch of states and returns Q-values of shape `(batch, n_actions)`. You can give it uint8 tensors directly.
- `ReplayBuffer(capacity)` has `add(s, a, r, s2, done)` and `sample(batch_size)`. `sample` returns numpy arrays in the order states, actions, rewards, next_states, dones. `len(buffer)` is the number of transitions stored.

In [ ]:
from environment import make_env
from network import DQN
from replay_buffer import ReplayBuffer


def test_environment_network_buffer():
    env = make_env(seed=0)
    state, _ = env.reset()
    assert state.shape == (4, 84, 84)
    assert state.dtype == np.uint8
    assert env.action_space.n == N_ACTIONS

    # Play 200 random steps and store them in the buffer
    buffer = ReplayBuffer(capacity=500)
    for _ in range(200):
        action = env.action_space.sample()
        next_state, reward, terminated, truncated, _ = env.step(action)
        buffer.add(state, action, reward, next_state, terminated)
        state = next_state
        if terminated or truncated:
            state, _ = env.reset()
    assert len(buffer) == 200

    # Sample a batch and pass it through the network
    states, actions, rewards, next_states, dones = buffer.sample(32)
    network = DQN(N_ACTIONS)
    q_values = network(torch.as_tensor(states))
    assert q_values.shape == (32, N_ACTIONS)
    env.close()


run_test(test_environment_network_buffer)

## 4. Epsilon-greedy, epsilon decay and CSV logging (HonourGod)
- `get_epsilon`: epsilon goes down in a straight line from `eps_start` to `eps_end` over `eps_decay` steps, then stays at `eps_end`.
- `select_action`: with probability `epsilon` pick a random action, otherwise pick the action with the highest Q-value.
- `log_episode`: add one row to the CSV at the end of each episode. Write the header first if the file doesn't exist yet. `plot_results.py` reads this file.

In [ ]:
def get_epsilon(step, eps_start, eps_end, eps_decay):
    # TODO (HonourGod): return epsilon for this step
    raise NotImplementedError


def select_action(state, online_net, epsilon, n_actions):
    # TODO (HonourGod): state is a (4, 84, 84) numpy array, return an int
    # The network expects a batch, so add a batch dimension and put it on `device`.
    raise NotImplementedError


def log_episode(csv_path, episode, step, reward, length, epsilon, loss):
    # TODO (HonourGod): append one row with the columns in CSV_COLUMNS
    raise NotImplementedError

In [ ]:
def test_get_epsilon():
    assert get_epsilon(0, 1.0, 0.02, 1000) == 1.0
    assert abs(get_epsilon(500, 1.0, 0.02, 1000) - 0.51) < 1e-6  # halfway down
    assert abs(get_epsilon(1000, 1.0, 0.02, 1000) - 0.02) < 1e-6  # reached eps_end
    assert abs(get_epsilon(5000, 1.0, 0.02, 1000) - 0.02) < 1e-6  # stays at eps_end


def test_select_action():
    network = DQN(N_ACTIONS).to(device)
    state = np.random.randint(0, 256, size=(4, 84, 84), dtype=np.uint8)

    # With epsilon = 0 the agent must always pick the best action
    with torch.no_grad():
        q_values = network(torch.as_tensor(state, device=device).unsqueeze(0))
    best_action = int(q_values.argmax())
    assert select_action(state, network, 0.0, N_ACTIONS) == best_action

    # With epsilon = 1 the agent must pick random actions
    actions = [select_action(state, network, 1.0, N_ACTIONS) for _ in range(200)]
    for action in actions:
        assert 0 <= action < N_ACTIONS
    assert len(set(actions)) > 1, "epsilon = 1 should give different actions"


def test_log_episode():
    path = Path("smoke_test/test_log.csv")
    path.parent.mkdir(exist_ok=True)
    if path.exists():
        path.unlink()

    log_episode(path, 1, 900, -21.0, 900, 0.99, 0.01)
    log_episode(path, 2, 1800, -20.0, 900, 0.98, 0.02)

    with open(path) as f:
        rows = list(csv.reader(f))
    assert rows[0] == CSV_COLUMNS  # header
    assert len(rows) == 3  # header + 2 episodes
    path.unlink()


run_test(test_get_epsilon)
run_test(test_select_action)
run_test(test_log_episode)

## 5. Learning update, target network and training loop (Anthony)
- `update_target`: copy the online network's weights into the target network.
- `learn`: sample a batch and get Q(s, a) from the online network. Use the target network (with no gradients) for the Bellman target `r + gamma * max Q_target(s', a') * (1 - done)`. Then compute the loss, backpropagate, take an optimizer step and return the loss as a float.
- `train(config)`: the main loop. It fills the buffer with `learning_starts` random steps, calls `learn` every `train_every` steps, updates the target every `target_update_every` steps, logs each episode with `log_episode`, and stops at `config["steps"]`. At the end it saves `online_net.state_dict()` to `config["out"]` and returns that path. The CSV goes next to it with the same name and a `.csv` extension.

In [ ]:
def update_target(online_net, target_net):
    # TODO (Anthony)
    raise NotImplementedError


def learn(online_net, target_net, buffer, optimizer, batch_size, gamma):
    # TODO (Anthony): one gradient step on a random batch, return the loss as a float
    # buffer.sample() gives numpy arrays, so convert them with torch.as_tensor(..., device=device)
    raise NotImplementedError


def train(config):
    # TODO (Anthony): run one experiment and return the path of the saved model
    raise NotImplementedError

In [ ]:
def test_update_target():
    online_net = DQN(N_ACTIONS)
    target_net = DQN(N_ACTIONS)
    update_target(online_net, target_net)

    online_weights = online_net.state_dict()
    target_weights = target_net.state_dict()
    for name in online_weights:
        assert torch.equal(online_weights[name], target_weights[name])


def test_learn():
    online_net = DQN(N_ACTIONS).to(device)
    target_net = copy.deepcopy(online_net)
    optimizer = torch.optim.Adam(online_net.parameters(), lr=1e-4)

    # Fill a small buffer with random transitions
    buffer = ReplayBuffer(capacity=100)
    for _ in range(100):
        state = np.random.randint(0, 256, size=(4, 84, 84), dtype=np.uint8)
        next_state = np.random.randint(0, 256, size=(4, 84, 84), dtype=np.uint8)
        action = random.randrange(N_ACTIONS)
        reward = random.choice([-1.0, 0.0, 1.0])
        buffer.add(state, action, reward, next_state, False)

    online_before = copy.deepcopy(online_net.state_dict())
    target_before = copy.deepcopy(target_net.state_dict())

    loss = learn(online_net, target_net, buffer, optimizer, batch_size=32, gamma=0.99)
    assert isinstance(loss, float)

    # The online network should change, the target network should not
    online_changed = False
    for name, weights in online_net.state_dict().items():
        if not torch.equal(weights, online_before[name]):
            online_changed = True
    assert online_changed, "learn() did not update the online network"

    for name, weights in target_net.state_dict().items():
        assert torch.equal(weights, target_before[name]), "learn() should not change the target network"


run_test(test_update_target)
run_test(test_learn)

## 6. Integration smoke test (everyone)
A 5,000-step run using all the parts together. It passes if the model file is saved, the CSV has the agreed columns, the loss is never `nan`, and the saved model loads the same way `play.py` loads it.

The reward won't improve in 5,000 steps. This run only checks that the parts work together.

In [ ]:
smoke_config = default_config.copy()
smoke_config["steps"] = 5_000
smoke_config["learning_starts"] = 1_000
smoke_config["buffer_size"] = 10_000


def test_integration():
    start = time.time()
    model_path = Path(train(smoke_config))
    print(f"Training took {time.time() - start:.0f} seconds")

    assert model_path.exists(), "no model file was saved"
    csv_path = model_path.with_suffix(".csv")
    assert csv_path.exists(), "no CSV log was written"

    log = pd.read_csv(csv_path)
    print(log)
    assert list(log.columns) == CSV_COLUMNS
    assert not log["loss"].isna().any(), "the loss has nan values"

    # Load the weights the same way play.py does
    network = DQN(N_ACTIONS)
    network.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))


run_test(test_integration)

## 7. Training plots (Alliane)
Set `csv_path` to the log of the run you want to plot.

In [ ]:
from plot_results import plot_training

csv_path = Path("smoke_test/model.csv")
if csv_path.exists():
    plot_training(pd.read_csv(csv_path), title=csv_path.stem)
    plt.show()
else:
    print(csv_path, "doesn't exist yet. Run the smoke test or an experiment first.")

## 8. Watching the agent play (Alliane)
The game window can't open in a notebook or on Colab, so run `play.py` from a terminal on your laptop:

```
uv run play.py                             # plays dqn_model.pth
uv run play.py --model Alliane/exp03.pth   # plays one experiment's model
uv run play.py --episodes 5
```